# Divergence comparison: KL vs chi^2 (vs Hellinger / TV) on the outlier simulation

The unbalanced penalty `tau * D_psi(rho || nu)` accepts any Csiszar divergence psi. This notebook
fits the uniform-weight UOT barycenter of the outlier mixtures with each psi (same tau), plus
balanced OT as the no-relaxation reference, and scores robustness (outlier contamination, distance
to the inlier modes) and distributional fit (MMD / W2 to the inlier-only ideal).

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt


def _bootstrap():
    here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
    # this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
    for d in (here, os.path.abspath(os.path.join(here, os.pardir, os.pardir, "tools"))):
        if d not in sys.path:
            sys.path.insert(0, d)
    import _repo
    _repo.add_paths()
    return _repo


P = _bootstrap()
REPO_ROOT = P.REPO
import uotreg as U
from uotreg.config import ModelConfig, TrainConfig, UOTConfig
from uotreg.metrics import mmd_rbf, w2, _HAS_POT
from uotreg.divergences import available_divergences
from uotreg import outlier_sim as od

print("uotreg", U.__version__, "| psi registry:", available_divergences(),
      "| W2 backend:", "EXACT (POT)" if _HAS_POT else "*** SLICED PROXY -- pip install pot ***")

## Parameters
`DIVERGENCES` = which psi to fit (KL + chi^2 are the point; Hellinger/TV are negative controls,
`INCLUDE_BAD=False` skips them). `TAU` is shared by all psi so the comparison is fair.

In [ ]:
DEVICE      = "auto"                 # "auto" (GPU if available) | "cpu"
# ----------------------------------------------------------------------------- SMOKE
# 1 = small and fast: runs end to end on a laptop. **NOT the paper's numbers.**
# 0 = the settings used in the paper.
SMOKE       = 1
# 1 = write results/figures to `new_results/`; 0 = keep everything in memory.
# The shipped `results/` tree is never modified either way.
SAVE = 0

TAU         = 1.0                    # shared unbalanced tolerance for ALL psi (fair comparison)
INCLUDE_BAD = True                   # also fit Hellinger + TV (the negative controls)
DIVERGENCES = ["kl", "chi2"] + (["hellinger", "tv"] if INCLUDE_BAD else [])
ADD_BALANCED = True                  # include balanced OT (no relaxation) as the "dragged" reference
N_CLOUDS    = 10
N_PER       = 600 if SMOKE else 2000
OUTER       = 6 if SMOKE else 41   # generator outer-iters (main budget)
N_SAMPLE    = 2000                   # points sampled from each fitted barycenter
SEED        = 0

## Data + the ideal inlier-only target
The reference sample is drawn from ONLY the inlier modes -- the distribution a perfectly robust
barycenter should match.

In [ ]:
clouds = od.generate_outlier_gmms(num=N_CLOUDS, n=N_PER, seed=SEED)
INLIER_MEANS = np.asarray(od.INLIER_MEANS, float); OUTLIER_MEANS = np.asarray(od.OUTLIER_MEANS, float)
WEIGHTS_FULL = np.asarray(od.WEIGHTS, float); DIM_ACTIVE, WITHIN_COV = od.DIM, 0.5
samplers = U.samplers_from_arrays(clouds, device=DEVICE)
weights = [1.0 / len(clouds)] * len(clouds)            # uniform-weight barycenter
pooled = np.concatenate(clouds, axis=0)


def contamination(X):
    """Fraction of points nearer an outlier mode than any inlier mode (lower = more robust)."""
    di = np.min(np.linalg.norm(X[:, None, :] - INLIER_MEANS[None], axis=2), axis=1)
    do = np.min(np.linalg.norm(X[:, None, :] - OUTLIER_MEANS[None], axis=2), axis=1)
    return float(np.mean(do < di))


def inlier_dist(X):
    """Mean distance of each point to its nearest inlier mode (lower = better)."""
    return float(np.min(np.linalg.norm(X[:, None, :] - INLIER_MEANS[None], axis=2), axis=1).mean())


def ideal_inlier_sample(n, seed=0):
    """Sample the inlier modes only (renormalized inlier weights) = the robust target."""
    rng = np.random.default_rng(seed)
    ni = len(INLIER_MEANS); w = WEIGHTS_FULL[:ni]; w = w / w.sum()
    comp = rng.choice(ni, size=n, p=w)
    L = np.linalg.cholesky(np.eye(DIM_ACTIVE) * WITHIN_COV)
    z = rng.standard_normal((n, DIM_ACTIVE)) @ L.T
    return (INLIER_MEANS[comp] + z).astype(np.float32)


target = ideal_inlier_sample(N_SAMPLE, seed=1)
print(f"outlier sim: {N_CLOUDS} clouds x {N_PER} pts in {DIM_ACTIVE}-D "
      f"({len(INLIER_MEANS)} inlier + "
      f"{len(OUTLIER_MEANS)} outlier modes) | psi = {DIVERGENCES}"
      + (" + balanced OT" if ADD_BALANCED else ""))

## Fit the barycenter for each psi (+ balanced OT)
Identical model / training / init for every run; only the divergence changes.

In [ ]:
model = ModelConfig(dim=DIM_ACTIVE, gen_hidden=256, gen_layers=4, gen_dropout=0.05,
                    map_hidden=100, map_layers=3, pot_hidden=100, pot_layers=3, dropout=0.05)


def make_train():
    return TrainConfig(outer_iters=OUTER,
                       d_iters=(10 if SMOKE else 50), t_iters=(3 if SMOKE else 10),
                       g_iters=(10 if SMOKE else 50), batch_size=64, batch_size_g=128,
                       lr_map=3e-4, lr_pot=3e-4, lr_gen=1e-4, weight_decay_td=1e-10,
                       weight_decay_gen=1e-8, device=DEVICE, verbose=False, seed=SEED)


def fit_barycenter(relaxation, divergence, tau=None):
    uc = UOTConfig(relaxation=relaxation, divergence=divergence, tau=(TAU if tau is None else tau))
    est = U.DistributionEstimator(model=model, train=make_train(), uot=uc)
    est.fit(samplers, weights=weights, init="gaussian",
            init_kwargs={"iters": (600 if SMOKE else 10000), "gaussian_scale": 6.0})
    return est.sample(N_SAMPLE), est


runs = []
if ADD_BALANCED:
    runs.append(("balanced OT", "balanced", "kl"))          # divergence unused when balanced
for dv in DIVERGENCES:
    runs.append((f"UOT {dv}", "one-sided", dv))

samples, estimators = {}, {}               # estimators kept so the fitted generator can be saved
for label, relaxation, dv in runs:
    t0 = time.time()
    samples[label], estimators[label] = fit_barycenter(relaxation, dv)
    print(f"  fit {label:14s} in {time.time()-t0:5.0f}s")

## Metrics (lower = better on all)
**contamination** = fraction of barycenter points nearer an outlier mode than any inlier mode;
**inlier-dist** = mean distance to the nearest inlier mode; **MMD / W2** = to the inlier-only ideal.

In [ ]:
if SMOKE:
    print("\n  !! SMOKE=1: under-trained -- every psi looks alike. The differences appear at SMOKE=0.")
print(f"\n[outlier sim, tau={TAU}, outer={OUTER}] psi comparison (lower=better):")
print(f"  {'method':16s}{'contamination':>15}{'inlier-dist':>13}{'MMD->ideal':>13}{'W2->ideal':>12}")
metrics = {}
for label in samples:
    X = samples[label]
    c = contamination(X); idst = inlier_dist(X)
    mm = mmd_rbf(X, target); ww = w2(X, target)
    metrics[label] = dict(contamination=c, inlier_dist=idst, mmd=mm, w2=ww)
    print(f"  {label:16s}{c:>15.3f}{idst:>13.3f}{mm:>13.3f}{ww:>12.3f}")

## Save (opt-in): sampled clouds, metrics, and the fitted generators -> `new_results/divergence`

In [ ]:
import json
RESULTS_W = P.results("divergence", write=True)
if SAVE:
    os.makedirs(RESULTS_W, exist_ok=True)
    np.savez(os.path.join(RESULTS_W, "outliers_samples.npz"),
             pooled=np.asarray(pooled, np.float32), target=np.asarray(target, np.float32),
             inlier_means=INLIER_MEANS.astype(np.float32), outlier_means=OUTLIER_MEANS.astype(np.float32),
             **{lab.replace(" ", "_"): np.asarray(X, np.float32) for lab, X in samples.items()})
    with open(os.path.join(RESULTS_W, "outliers_metrics.json"), "w") as f:
        json.dump({lab: {k: float(v) for k, v in metrics[lab].items()} for lab in metrics}, f, indent=2)
    for lab, est in estimators.items():
        est.save(os.path.join(RESULTS_W, f"G_outlier_{lab.replace(' ', '')}.pth"))
    print(f"saved samples/metrics/{len(estimators)} generators -> {RESULTS_W}")
else:
    print("SAVE=0 -- results kept in memory (set SAVE=1 to write new_results/)")

## Figure: barycenter by divergence

In [ ]:
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11, "axes.labelsize": 11,
                     "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10})
N_BG, N_BARY = 2500, 1200          # points DRAWN: N_BG = grey background (pooled), N_BARY = barycenter cloud
Pp = PCA(2).fit(pooled).transform
_vrng = np.random.default_rng(0)
BG = Pp(pooled[_vrng.choice(len(pooled), min(N_BG, len(pooled)), replace=False)])
IM, OM, BARY_C = Pp(INLIER_MEANS), Pp(OUTLIER_MEANS), "#7b3fa0"
labels = list(samples.keys())
fig, axes = plt.subplots(1, len(labels), figsize=(3.6 * len(labels), 4.0), dpi=140, sharex=True, sharey=True)
for ax, label in zip(np.atleast_1d(axes), labels):
    _Xb = np.asarray(samples[label]); _Xb = _Xb[_vrng.choice(len(_Xb), min(N_BARY, len(_Xb)), replace=False)]
    Xp = Pp(_Xb)
    ax.scatter(BG[:, 0], BG[:, 1], s=5, c="0.85", alpha=0.4)
    ax.scatter(Xp[:, 0], Xp[:, 1], s=6, c=BARY_C, alpha=0.55)
    ax.scatter(IM[:, 0], IM[:, 1], marker="*", s=230, c="#08306b", edgecolors="w", linewidths=0.8, zorder=6)
    ax.scatter(OM[:, 0], OM[:, 1], marker="X", s=130, c="#67000d", edgecolors="w", linewidths=0.8, zorder=6)
    ax.set_title(f"{label}\nW2 to ideal = {metrics[label]['w2']:.2f}", fontsize=11); ax.set_xlabel("PC1")
np.atleast_1d(axes)[0].set_ylabel("PC2")
np.atleast_1d(axes)[0].legend(handles=[
    Line2D([0], [0], marker="o", ls="", mfc=BARY_C, mec="none", label="barycenter"),
    Line2D([0], [0], marker="*", ls="", mfc="#08306b", mec="w", markersize=12, label="inlier modes"),
    Line2D([0], [0], marker="X", ls="", mfc="#67000d", mec="w", markersize=9, label="outlier modes")],
    loc="upper right")
fig.suptitle(f"Outlier simulation: barycenter by divergence (tau={TAU}) -- mass should concentrate on the inlier modes",
             fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()